### Version: (1,1,2,2), heavy transform - peak 78.5%


In [ ]:
# Code: 


YOUR_MEAN = 0.1156
YOUR_STD = 0.2198


train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.95,1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[YOUR_MEAN], std=[YOUR_STD])
])


test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[YOUR_MEAN], std=[YOUR_STD])
])


# load data
train_dataset = datasets.ImageFolder(root="data/ADNI/AD_NC/train", transform=train_transform)
test_dataset  = datasets.ImageFolder(root="data/ADNI/AD_NC/test",  transform=test_transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=1)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=1)

print(f"Train length: {len(train_dataset)}, Test length: {len(test_dataset)}")
print(train_dataset[0][0].mean(), train_dataset[0][0].std())


class SmallBlock(nn.Module):
    def __init__(self, dim, layer_scale_init_value=1e-5):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        # We'll use channels_last LayerNorm via explicit permute
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pw1 = nn.Linear(dim, 4*dim)
        self.act = nn.GELU()
        self.pw2 = nn.Linear(4*dim, dim)
        if layer_scale_init_value > 0:
            self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((dim)), requires_grad=True)
        else:
            self.gamma = None

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)                          # N,C,H,W
        x = x.permute(0, 2, 3, 1)                   # N,H,W,C
        x = self.norm(x)
        x = self.pw1(x)
        x = self.act(x)
        x = self.pw2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)                   # N,C,H,W
        return shortcut + x

class MiniConvNeXt(nn.Module):
    def __init__(self, in_chans=1, num_classes=2,
                 depths=(1,1,2,2), dims=(32,64,128,256), layer_scale_init_value=1e-5):
        super().__init__()
        assert len(depths)==4 and len(dims)==4

        self.dropout = nn.Dropout(p=0.25)
        # stem
        self.downsamples = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            # Use LayerNorm over channels_first by wrapping below in forward_features
        )
        self.downsamples.append(stem)

        for i in range(3):
            self.downsamples.append(
                nn.Sequential(
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
                )
            )

        # stages
        self.stages = nn.ModuleList()
        for i in range(4):
            blocks = []
            for _ in range(depths[i]):
                blocks.append(SmallBlock(dims[i], layer_scale_init_value=layer_scale_init_value))
            self.stages.append(nn.Sequential(*blocks))

        # final norm and head
        self.final_norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                trunc_normal_(m.weight, std=.02)
                if getattr(m, "bias", None) is not None:
                    nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        # x: N, C, H, W  (C=1)
        for i in range(4):
            x = self.downsamples[i](x)   # conv downsample (N,C,H,W)
            # pass each stage
            x = self.stages[i](x)
        # global pool
        x = x.mean([-2, -1])            # N, C
        x = self.final_norm(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.dropout(x)
        x = self.head(x)
        return x

# set up model and parameters


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not torch.cuda.is_available():
    print("Warning CUDA not Found. Using CPU")    
model = MiniConvNeXt(in_chans=1, num_classes=2,
                 depths=(1,1,2,2), dims=(32,64,128,256)).to(device)
    
    
EPOCHS = 200
    
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)  # AdamW is preferable
# One-cycle LR often helps for from-scratch training:


from torch.optim.lr_scheduler import OneCycleLR
#scheduler = OneCycleLR(optimizer, max_lr=3e-3, steps_per_epoch=len(train_loader), epochs=EPOCHS)

optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=5e-3,
                       steps_per_epoch=len(train_loader),
                       epochs=EPOCHS)

scaler = torch.amp.GradScaler(enabled=False)

# TRAINGIN LOOP

print("\n Beginning Training:")
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    running_loss = 0.0
    image_count = 0
    prev_100_images_start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        #  Forward pass in mixed precision
        #with torch.amp.autocast(device_type="cuda"):
        outputs = model(images)
        loss = criterion(outputs, labels)

        #  Scaled backward pass
        scaler.scale(loss).backward()
        
        if not torch.isfinite(loss):
            print("Non-finite loss, stopping training")
            break

        #  Unscale + clip gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)

        #  Step with scaler
        scaler.step(optimizer)
        scaler.update()

        #  Step LR scheduler once per batch
        scheduler.step()

        running_loss += loss.item()
        image_count += images.size(0)

        # Occasional progress report
        if image_count % 5000 == 0:
            print(f"Trained {image_count} images, Time: {time.time() - prev_100_images_start:.1f}s")
            prev_100_images_start = time.time()

    avg_loss = running_loss / len(train_loader)


    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()


    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | Test Accuracy: {100 * correct / total:.2f}% | Time: {time.time()-epoch_start:.1f}s")


print(f"Finished Training on {image_count} images in {time.time() - start_time} seconds")

In [ ]:
# Results
 Beginning Training:
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 01/200 | Loss: 0.6849 | Test Accuracy: 56.98% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.7s
Epoch 02/200 | Loss: 0.6591 | Test Accuracy: 59.13% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 03/200 | Loss: 0.6244 | Test Accuracy: 64.39% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 04/200 | Loss: 0.6137 | Test Accuracy: 66.32% | Time: 68.2s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.2s
Epoch 05/200 | Loss: 0.5963 | Test Accuracy: 64.79% | Time: 68.8s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 06/200 | Loss: 0.5948 | Test Accuracy: 67.27% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 07/200 | Loss: 0.5794 | Test Accuracy: 67.76% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.3s
Epoch 08/200 | Loss: 0.5823 | Test Accuracy: 68.33% | Time: 67.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 09/200 | Loss: 0.5814 | Test Accuracy: 68.67% | Time: 68.0s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 10/200 | Loss: 0.5819 | Test Accuracy: 68.72% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.3s
Epoch 11/200 | Loss: 0.5748 | Test Accuracy: 65.62% | Time: 68.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 12/200 | Loss: 0.5704 | Test Accuracy: 70.06% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 13/200 | Loss: 0.5578 | Test Accuracy: 65.58% | Time: 66.7s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 14/200 | Loss: 0.5681 | Test Accuracy: 69.41% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 15/200 | Loss: 0.5550 | Test Accuracy: 60.28% | Time: 67.3s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 25.0s
Epoch 16/200 | Loss: 0.5482 | Test Accuracy: 68.41% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.6s
Epoch 17/200 | Loss: 0.5413 | Test Accuracy: 68.06% | Time: 68.7s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.4s
Epoch 18/200 | Loss: 0.5482 | Test Accuracy: 70.16% | Time: 68.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 19/200 | Loss: 0.5330 | Test Accuracy: 67.68% | Time: 67.4s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 20/200 | Loss: 0.5286 | Test Accuracy: 59.81% | Time: 67.3s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.6s
Epoch 21/200 | Loss: 0.5161 | Test Accuracy: 70.36% | Time: 67.7s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.7s
Epoch 22/200 | Loss: 0.5103 | Test Accuracy: 71.70% | Time: 67.1s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.8s
Epoch 23/200 | Loss: 0.5015 | Test Accuracy: 65.08% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 24/200 | Loss: 0.4874 | Test Accuracy: 69.51% | Time: 67.2s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 25/200 | Loss: 0.4832 | Test Accuracy: 64.41% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 26/200 | Loss: 0.4852 | Test Accuracy: 67.78% | Time: 67.8s
Trained 10000 images, Time: 25.7s
Trained 20000 images, Time: 25.3s
Epoch 27/200 | Loss: 0.4792 | Test Accuracy: 64.18% | Time: 68.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 28/200 | Loss: 0.4702 | Test Accuracy: 65.67% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 29/200 | Loss: 0.4630 | Test Accuracy: 73.27% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 30/200 | Loss: 0.4563 | Test Accuracy: 71.70% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.5s
Epoch 31/200 | Loss: 0.4528 | Test Accuracy: 72.02% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 32/200 | Loss: 0.4498 | Test Accuracy: 72.22% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.6s
Epoch 33/200 | Loss: 0.4357 | Test Accuracy: 69.94% | Time: 67.5s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 34/200 | Loss: 0.4409 | Test Accuracy: 65.36% | Time: 67.4s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 35/200 | Loss: 0.4365 | Test Accuracy: 72.70% | Time: 66.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.3s
Epoch 36/200 | Loss: 0.4258 | Test Accuracy: 70.12% | Time: 67.5s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.1s
Epoch 37/200 | Loss: 0.4187 | Test Accuracy: 74.01% | Time: 67.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 38/200 | Loss: 0.4170 | Test Accuracy: 72.98% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 39/200 | Loss: 0.4109 | Test Accuracy: 72.59% | Time: 66.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 40/200 | Loss: 0.4036 | Test Accuracy: 66.10% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 41/200 | Loss: 0.3950 | Test Accuracy: 71.87% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 42/200 | Loss: 0.3937 | Test Accuracy: 69.07% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 43/200 | Loss: 0.3883 | Test Accuracy: 68.84% | Time: 66.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 44/200 | Loss: 0.3803 | Test Accuracy: 71.58% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 45/200 | Loss: 0.3798 | Test Accuracy: 68.50% | Time: 67.5s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 24.8s
Epoch 46/200 | Loss: 0.3763 | Test Accuracy: 70.84% | Time: 67.7s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.9s
Epoch 47/200 | Loss: 0.3665 | Test Accuracy: 73.68% | Time: 66.9s
Trained 10000 images, Time: 25.7s
Trained 20000 images, Time: 24.9s
Epoch 48/200 | Loss: 0.3651 | Test Accuracy: 71.12% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 49/200 | Loss: 0.3550 | Test Accuracy: 74.02% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 50/200 | Loss: 0.3562 | Test Accuracy: 74.24% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 51/200 | Loss: 0.3457 | Test Accuracy: 72.63% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 52/200 | Loss: 0.3384 | Test Accuracy: 76.00% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 53/200 | Loss: 0.3324 | Test Accuracy: 72.29% | Time: 67.1s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.7s
Epoch 54/200 | Loss: 0.3226 | Test Accuracy: 71.94% | Time: 66.8s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.0s
Epoch 55/200 | Loss: 0.3218 | Test Accuracy: 71.12% | Time: 68.1s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.9s
Epoch 56/200 | Loss: 0.3154 | Test Accuracy: 73.60% | Time: 67.0s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 57/200 | Loss: 0.3036 | Test Accuracy: 74.14% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.6s
Epoch 58/200 | Loss: 0.3023 | Test Accuracy: 72.44% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 59/200 | Loss: 0.3020 | Test Accuracy: 75.68% | Time: 67.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 60/200 | Loss: 0.2884 | Test Accuracy: 73.40% | Time: 66.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.6s
Epoch 61/200 | Loss: 0.2879 | Test Accuracy: 72.58% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 62/200 | Loss: 0.2818 | Test Accuracy: 73.71% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 63/200 | Loss: 0.2728 | Test Accuracy: 75.44% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.2s
Epoch 64/200 | Loss: 0.2721 | Test Accuracy: 73.58% | Time: 67.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 65/200 | Loss: 0.2627 | Test Accuracy: 73.51% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 66/200 | Loss: 0.2536 | Test Accuracy: 73.66% | Time: 67.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 67/200 | Loss: 0.2518 | Test Accuracy: 74.41% | Time: 67.1s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.7s
Epoch 68/200 | Loss: 0.2478 | Test Accuracy: 74.79% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 69/200 | Loss: 0.2398 | Test Accuracy: 76.78% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 70/200 | Loss: 0.2348 | Test Accuracy: 73.68% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 71/200 | Loss: 0.2229 | Test Accuracy: 75.62% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 72/200 | Loss: 0.2260 | Test Accuracy: 74.22% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 73/200 | Loss: 0.2209 | Test Accuracy: 75.24% | Time: 67.3s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.9s
Epoch 74/200 | Loss: 0.2190 | Test Accuracy: 75.81% | Time: 67.4s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 75/200 | Loss: 0.2126 | Test Accuracy: 74.63% | Time: 66.9s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.6s
Epoch 76/200 | Loss: 0.2081 | Test Accuracy: 73.02% | Time: 67.4s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 77/200 | Loss: 0.2022 | Test Accuracy: 75.50% | Time: 67.5s
Trained 10000 images, Time: 25.9s
Trained 20000 images, Time: 25.1s
Epoch 78/200 | Loss: 0.1976 | Test Accuracy: 73.06% | Time: 68.4s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.4s
Epoch 79/200 | Loss: 0.1976 | Test Accuracy: 75.47% | Time: 68.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 80/200 | Loss: 0.1897 | Test Accuracy: 77.33% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 81/200 | Loss: 0.1857 | Test Accuracy: 74.31% | Time: 67.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 82/200 | Loss: 0.1849 | Test Accuracy: 76.78% | Time: 67.3s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.5s
Epoch 83/200 | Loss: 0.1780 | Test Accuracy: 76.13% | Time: 68.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.5s
Epoch 84/200 | Loss: 0.1729 | Test Accuracy: 72.80% | Time: 66.9s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 85/200 | Loss: 0.1672 | Test Accuracy: 75.69% | Time: 66.9s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.7s
Epoch 86/200 | Loss: 0.1685 | Test Accuracy: 77.23% | Time: 66.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 87/200 | Loss: 0.1658 | Test Accuracy: 78.11% | Time: 67.0s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 88/200 | Loss: 0.1643 | Test Accuracy: 76.14% | Time: 67.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 89/200 | Loss: 0.1578 | Test Accuracy: 76.39% | Time: 67.3s
Trained 10000 images, Time: 25.6s
Trained 20000 images, Time: 24.9s
Epoch 90/200 | Loss: 0.1576 | Test Accuracy: 76.54% | Time: 69.2s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 91/200 | Loss: 0.1515 | Test Accuracy: 75.43% | Time: 68.2s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 25.2s
Epoch 92/200 | Loss: 0.1538 | Test Accuracy: 73.23% | Time: 68.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 93/200 | Loss: 0.1447 | Test Accuracy: 75.97% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 94/200 | Loss: 0.1375 | Test Accuracy: 77.32% | Time: 67.4s
Trained 10000 images, Time: 24.6s
Trained 20000 images, Time: 24.6s
Epoch 95/200 | Loss: 0.1419 | Test Accuracy: 77.80% | Time: 66.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 96/200 | Loss: 0.1391 | Test Accuracy: 73.89% | Time: 67.5s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 97/200 | Loss: 0.1344 | Test Accuracy: 77.21% | Time: 67.3s
Trained 10000 images, Time: 25.5s
Trained 20000 images, Time: 24.9s
Epoch 98/200 | Loss: 0.1284 | Test Accuracy: 78.19% | Time: 67.6s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.0s
Epoch 99/200 | Loss: 0.1251 | Test Accuracy: 77.74% | Time: 67.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 100/200 | Loss: 0.1260 | Test Accuracy: 76.76% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 101/200 | Loss: 0.1228 | Test Accuracy: 75.74% | Time: 67.3s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.5s
Epoch 102/200 | Loss: 0.1194 | Test Accuracy: 74.67% | Time: 68.5s
Trained 10000 images, Time: 25.8s
Trained 20000 images, Time: 25.2s
Epoch 103/200 | Loss: 0.1183 | Test Accuracy: 74.14% | Time: 68.7s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 104/200 | Loss: 0.1231 | Test Accuracy: 75.39% | Time: 67.9s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 105/200 | Loss: 0.1167 | Test Accuracy: 77.43% | Time: 67.1s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 106/200 | Loss: 0.1087 | Test Accuracy: 74.50% | Time: 67.9s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 107/200 | Loss: 0.1115 | Test Accuracy: 77.97% | Time: 66.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 108/200 | Loss: 0.1087 | Test Accuracy: 77.13% | Time: 67.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 109/200 | Loss: 0.1050 | Test Accuracy: 76.74% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.7s
Epoch 110/200 | Loss: 0.1010 | Test Accuracy: 76.83% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.3s
Epoch 111/200 | Loss: 0.1004 | Test Accuracy: 77.77% | Time: 68.1s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 112/200 | Loss: 0.0980 | Test Accuracy: 75.32% | Time: 67.6s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.7s
Epoch 113/200 | Loss: 0.0937 | Test Accuracy: 76.48% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 114/200 | Loss: 0.0941 | Test Accuracy: 75.41% | Time: 68.0s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.4s
Epoch 115/200 | Loss: 0.0923 | Test Accuracy: 76.43% | Time: 69.1s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.1s
Epoch 116/200 | Loss: 0.0850 | Test Accuracy: 75.92% | Time: 67.9s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.4s
Epoch 117/200 | Loss: 0.0888 | Test Accuracy: 78.08% | Time: 68.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.3s
Epoch 118/200 | Loss: 0.0920 | Test Accuracy: 77.28% | Time: 68.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 119/200 | Loss: 0.0836 | Test Accuracy: 75.10% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.0s
Epoch 120/200 | Loss: 0.0796 | Test Accuracy: 77.96% | Time: 68.0s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.2s
Epoch 121/200 | Loss: 0.0821 | Test Accuracy: 74.54% | Time: 68.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 122/200 | Loss: 0.0720 | Test Accuracy: 77.38% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.6s
Epoch 123/200 | Loss: 0.0724 | Test Accuracy: 75.54% | Time: 67.8s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 124/200 | Loss: 0.0763 | Test Accuracy: 77.10% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 125/200 | Loss: 0.0702 | Test Accuracy: 77.97% | Time: 67.1s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.7s
Epoch 126/200 | Loss: 0.0705 | Test Accuracy: 78.01% | Time: 67.5s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.7s
Epoch 127/200 | Loss: 0.0700 | Test Accuracy: 75.14% | Time: 69.2s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.9s
Epoch 128/200 | Loss: 0.0708 | Test Accuracy: 76.60% | Time: 67.4s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.0s
Epoch 129/200 | Loss: 0.0665 | Test Accuracy: 77.86% | Time: 68.2s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.4s
Epoch 130/200 | Loss: 0.0611 | Test Accuracy: 77.31% | Time: 68.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 131/200 | Loss: 0.0641 | Test Accuracy: 78.14% | Time: 67.1s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 132/200 | Loss: 0.0604 | Test Accuracy: 77.41% | Time: 67.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 133/200 | Loss: 0.0612 | Test Accuracy: 77.40% | Time: 67.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 134/200 | Loss: 0.0622 | Test Accuracy: 77.13% | Time: 67.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 135/200 | Loss: 0.0567 | Test Accuracy: 77.78% | Time: 67.6s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 136/200 | Loss: 0.0551 | Test Accuracy: 76.96% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 137/200 | Loss: 0.0534 | Test Accuracy: 75.98% | Time: 67.2s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 25.2s
Epoch 138/200 | Loss: 0.0525 | Test Accuracy: 75.58% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 139/200 | Loss: 0.0513 | Test Accuracy: 76.71% | Time: 67.9s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 140/200 | Loss: 0.0461 | Test Accuracy: 77.02% | Time: 67.6s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.2s
Epoch 141/200 | Loss: 0.0494 | Test Accuracy: 77.33% | Time: 67.4s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 142/200 | Loss: 0.0447 | Test Accuracy: 76.60% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.0s
Epoch 143/200 | Loss: 0.0430 | Test Accuracy: 77.47% | Time: 67.3s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.7s
Epoch 144/200 | Loss: 0.0424 | Test Accuracy: 77.64% | Time: 67.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.0s
Epoch 145/200 | Loss: 0.0443 | Test Accuracy: 78.11% | Time: 67.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 146/200 | Loss: 0.0427 | Test Accuracy: 76.01% | Time: 68.0s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 147/200 | Loss: 0.0400 | Test Accuracy: 78.51% | Time: 67.6s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 148/200 | Loss: 0.0390 | Test Accuracy: 76.91% | Time: 68.2s
Trained 10000 images, Time: 25.9s
Trained 20000 images, Time: 24.8s
Epoch 149/200 | Loss: 0.0379 | Test Accuracy: 78.28% | Time: 68.1s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.6s
Epoch 150/200 | Loss: 0.0362 | Test Accuracy: 76.77% | Time: 66.8s
Trained 10000 images, Time: 24.7s
Trained 20000 images, Time: 24.8s
Epoch 151/200 | Loss: 0.0362 | Test Accuracy: 76.43% | Time: 66.9s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 152/200 | Loss: 0.0368 | Test Accuracy: 76.89% | Time: 67.7s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 25.1s
Epoch 153/200 | Loss: 0.0329 | Test Accuracy: 77.79% | Time: 67.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 154/200 | Loss: 0.0357 | Test Accuracy: 76.80% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 155/200 | Loss: 0.0330 | Test Accuracy: 76.98% | Time: 67.7s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 156/200 | Loss: 0.0310 | Test Accuracy: 78.28% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 157/200 | Loss: 0.0313 | Test Accuracy: 77.71% | Time: 67.7s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.4s
Epoch 158/200 | Loss: 0.0270 | Test Accuracy: 76.88% | Time: 67.8s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 159/200 | Loss: 0.0294 | Test Accuracy: 76.70% | Time: 67.2s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.8s
Epoch 160/200 | Loss: 0.0279 | Test Accuracy: 77.53% | Time: 67.3s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 161/200 | Loss: 0.0271 | Test Accuracy: 76.32% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 162/200 | Loss: 0.0257 | Test Accuracy: 76.92% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 163/200 | Loss: 0.0242 | Test Accuracy: 76.61% | Time: 67.3s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.7s
Epoch 164/200 | Loss: 0.0247 | Test Accuracy: 77.24% | Time: 66.9s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.6s
Epoch 165/200 | Loss: 0.0214 | Test Accuracy: 78.00% | Time: 67.0s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.3s
Epoch 166/200 | Loss: 0.0209 | Test Accuracy: 77.00% | Time: 67.9s
Trained 10000 images, Time: 26.2s
Trained 20000 images, Time: 25.2s
Epoch 167/200 | Loss: 0.0208 | Test Accuracy: 77.40% | Time: 69.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 24.9s
Epoch 168/200 | Loss: 0.0220 | Test Accuracy: 77.31% | Time: 67.7s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 25.3s
Epoch 169/200 | Loss: 0.0212 | Test Accuracy: 77.19% | Time: 68.0s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 24.8s
Epoch 170/200 | Loss: 0.0217 | Test Accuracy: 76.91% | Time: 67.6s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.1s
Epoch 171/200 | Loss: 0.0203 | Test Accuracy: 77.03% | Time: 67.6s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 172/200 | Loss: 0.0180 | Test Accuracy: 78.49% | Time: 67.7s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.9s
Epoch 173/200 | Loss: 0.0174 | Test Accuracy: 77.32% | Time: 67.5s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.7s
Epoch 174/200 | Loss: 0.0178 | Test Accuracy: 77.80% | Time: 67.4s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 24.7s
Epoch 175/200 | Loss: 0.0161 | Test Accuracy: 76.81% | Time: 67.5s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.0s
Epoch 176/200 | Loss: 0.0167 | Test Accuracy: 77.57% | Time: 67.9s
Trained 10000 images, Time: 26.0s
Trained 20000 images, Time: 25.6s
Epoch 177/200 | Loss: 0.0138 | Test Accuracy: 77.29% | Time: 69.2s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 24.8s
Epoch 178/200 | Loss: 0.0148 | Test Accuracy: 76.64% | Time: 67.4s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.9s
Epoch 179/200 | Loss: 0.0153 | Test Accuracy: 77.56% | Time: 67.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 180/200 | Loss: 0.0149 | Test Accuracy: 77.11% | Time: 67.8s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.0s
Epoch 181/200 | Loss: 0.0146 | Test Accuracy: 77.56% | Time: 67.9s
Trained 10000 images, Time: 25.1s
Trained 20000 images, Time: 25.1s
Epoch 182/200 | Loss: 0.0130 | Test Accuracy: 77.73% | Time: 67.5s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.8s
Epoch 183/200 | Loss: 0.0126 | Test Accuracy: 77.01% | Time: 67.8s
Trained 10000 images, Time: 25.2s
Trained 20000 images, Time: 25.1s
Epoch 184/200 | Loss: 0.0133 | Test Accuracy: 77.78% | Time: 67.9s
Trained 10000 images, Time: 25.3s
Trained 20000 images, Time: 24.8s
Epoch 185/200 | Loss: 0.0119 | Test Accuracy: 77.18% | Time: 68.2s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 25.4s
Epoch 186/200 | Loss: 0.0141 | Test Accuracy: 77.70% | Time: 68.3s
Trained 10000 images, Time: 24.8s
Trained 20000 images, Time: 24.8s
Epoch 187/200 | Loss: 0.0106 | Test Accuracy: 77.27% | Time: 67.5s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.3s
Epoch 188/200 | Loss: 0.0120 | Test Accuracy: 77.57% | Time: 67.7s
Trained 10000 images, Time: 25.0s
Trained 20000 images, Time: 25.2s
Epoch 189/200 | Loss: 0.0135 | Test Accuracy: 77.18% | Time: 67.8s
Trained 10000 images, Time: 24.9s
Trained 20000 images, Time: 24.9s
Epoch 190/200 | Loss: 0.0104 | Test Accuracy: 77.43% | Time: 67.5s
Trained 10000 images, Time: 25.4s
Trained 20000 images, Time: 25.2s
Epoch 191/200 | Loss: 0.0098 | Test Accuracy: 77.47% | Time: 68.2s
Trained 10000 images, Time: 25.1s